In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [6]:
from transformers import AutoModel, AutoTokenizer, DataCollatorForLanguageModeling, TrainingArguments, AutoModelForCausalLM, Trainer
from datasets import load_dataset
import torch, gc
import evaluate
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict
from tqdm import tqdm
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from google.colab import drive

In [7]:
def preprocess_function(example, tokenizer, max_length=512):
    messages = [
        {
            "role": "system",
            "content": f"You are a medical assistant. Specialty: {example['focus_area']}"
        },
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    # Apply Qwen2 chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors=None
    )

    input_ids = tokenized["input_ids"]
    labels = input_ids.copy()

    assistant_start = text.rfind(example["answer"])
    if assistant_start != -1:
        prefix_text = text[:assistant_start]
        prefix_ids = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
        for i in range(min(len(prefix_ids), len(labels))):
            labels[i] = -100

    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    labels[:len(prefix_ids)] = [-100] * len(prefix_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": tokenized["attention_mask"],
        "labels": labels
    }

In [9]:
checkpoints = "Qwen/Qwen2.5-0.5B"
model = AutoModelForCausalLM.from_pretrained(checkpoints)
tokenizer = AutoTokenizer.from_pretrained(checkpoints)
print(f"Model type: {type(model)}")  # Should show CausalLM in the name
print(f"Model config: {model.config.architectures}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Model config: ['Qwen2ForCausalLM']


In [ ]:
dataset = load_dataset("csv", data_files={
    "train": "/content/drive/MyDrive/Colab Notebooks/chatbot_project/dataset/medquad_train.csv",
    "validation": "/content/drive/MyDrive/Colab Notebooks/chatbot_project/dataset/medquad_validation.csv"
})
dataset

FileNotFoundError: Unable to find '/content/nlp\data\processed\medquad_train.csv'

In [ ]:
tokenized_dataset = dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    remove_columns=dataset["train"].column_names,
    batched=False,
    desc="Tokenizing dataset"
)

tokenized_dataset = tokenized_dataset.filter(
    lambda x: len(x["input_ids"]) > 0,
    desc="Filtering empty examples"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8  # For efficiency
)

In [ ]:
@dataclass
class DataCollatorForCausalLM:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Find max length in batch
        max_length = max(len(f["input_ids"]) for f in features)
        
        batch = {
            "input_ids": [],
            "attention_mask": [],
            "labels": []
        }
        
        for feature in features:
            input_ids = feature["input_ids"]
            attention_mask = feature["attention_mask"]
            labels = feature["labels"]
            
            # Calculate padding length
            padding_length = max_length - len(input_ids)
            
            # Pad from the right
            padded_input_ids = input_ids + [self.tokenizer.pad_token_id] * padding_length
            padded_attention_mask = attention_mask + [0] * padding_length
            padded_labels = labels + [-100] * padding_length
            
            batch["input_ids"].append(padded_input_ids)
            batch["attention_mask"].append(padded_attention_mask)
            batch["labels"].append(padded_labels)
        
        # Convert to tensors
        return {
            "input_ids": torch.tensor(batch["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(batch["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(batch["labels"], dtype=torch.long)
        }

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("qwen2-medical-qa")

In [ ]:
data_collator = DataCollatorForCausalLM(tokenizer=tokenizer)

# Training arguments
training_args = TrainingArguments(
    output_dir="./qwen2-medical-qa",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    num_train_epochs=1,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    load_best_model_at_end=True,
    report_to="mlflow",     
    run_name="qwen2-medical-qa-run",
    remove_unused_columns=False,
)


# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

# Train
trainer.train()

# Save
trainer.save_model("./qwen2-medical-qa/final")
tokenizer.save_pretrained("./qwen2-medical-qa/final")
mlflow.log_artifacts("./qwen2-medical-qa/final", artifact_path="model")

In [ ]:
# Evaluation metrics function

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=True
    )
    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True
    )

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    rouge_scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    exact_match = np.mean([
        int(p.lower() == l.lower())
        for p, l in zip(decoded_preds, decoded_labels)
    ])

    return {
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
        "exact_match": exact_match
    }


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "./qwen2-medical-qa/final",
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "./qwen2-medical-qa/final",
    trust_remote_code=True
)
eval_args = TrainingArguments(
    output_dir="./qwen2-medical-qa-eval",
    per_device_eval_batch_size=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
torch.cuda.empty_cache()
metrics = trainer.evaluate()
print(metrics)